# ¿Un Whisper más rápido para la revisión?

La revisión de la fábrica no toca el audio: solo decide qué frases se repiten. Este cuaderno comprueba, con frases ya
generadas, que una configuración más rápida de Whisper decide igual que la de siempre (**large-v3 con marcas por
palabra**, cuya decisión está en el registro de cada episodio) y caza las averías que la revisión existe para cazar
(una cola de más al final, una frase cortada). Lógica en `fabrica/comparar_whisper.py`.

Prueba tres configuraciones, cada una en su propio proceso (la GPU queda limpia entre una y otra):
**turbo con marcas por segmento**, **large-v3 con marcas por segmento** y **turbo con marcas por palabra**.

**GPU: T4.** Unos 15–20 min (~0,6 unidades). No genera audio ni toca los episodios: deja el resultado en
`naval-podcast/pruebas/whisper_turbo.md` y desconecta. Ejecuta las celdas 1 a 6 seguidas («Ejecutar todo»).

In [ ]:
# 1. Instalación (no hace falta reiniciar)
!pip install -q jiwer num2words

In [ ]:
# 2. Parámetros: de qué episodios ya generados se toman las frases, y qué configuraciones se prueban
EPISODIOS = ["2019/018-skill-business", "2019/027-hourly-rate", "2019/034-reject-advice", "2019/041-envy",
             "2019/051-angel-1", "2020/068-kapil"]
N_FRASES = 200    # frases reales (todas las sospechosas entran siempre)
N_AVERIAS = 60    # averías fabricadas: la mitad con cola de más, la mitad cortadas
CONFIGURACIONES = [   # (nombre, modelo, marcas, frases a la vez)
    ("turbo · segmento", "openai/whisper-large-v3-turbo", "segmento", 16),
    ("large-v3 · segmento", "openai/whisper-large-v3", "segmento", 16),
    ("turbo · palabra", "openai/whisper-large-v3-turbo", "palabra", 2),
]

In [ ]:
# 3. Preparar: código, Drive y las frases con su audio (sin GPU)
import os, sys, subprocess, json, time, pickle
from pathlib import Path
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path[:0] = ["/content/repo/fabrica", "/content/repo/piloto", "/content/repo/criba"]
import comun, fabrica, medir, comparar_whisper as cw
comun.montar_drive()
comun.desconectar_si_falla()
CFG = fabrica.cargar_config()
RAIZ_EPISODIOS = Path(CFG["rutas"]["drive"]) / CFG["rutas"]["episodios"]
carpetas = []
for ep in EPISODIOS:
    base = Path("/content/repo/episodios") / ep
    titulo_en = json.loads((base / "metadata.en.json").read_text(encoding="utf-8"))["titulo"]
    c = fabrica.carpeta_episodio(RAIZ_EPISODIOS, ep, titulo_en, crear=False)
    if (c / "registro_tts.jsonl").exists():
        carpetas.append(c)
    else:
        print("sin registro, lo salto:", ep)
ITEMS = cw.frases_de(carpetas, N_FRASES)
for i in ITEMS:
    i["audio"] = cw.audio_16k(i["ruta"])
ITEMS = cw.con_averias(ITEMS, N_AVERIAS)
Path("/content/items_whisper.pkl").write_bytes(pickle.dumps(ITEMS))
reales = sum(i["tipo"] == "real" for i in ITEMS)
print(f"{reales} frases reales ({sum(i['ok_registro'] is False for i in ITEMS if i['tipo'] == 'real')} sospechosas) "
      f"y {len(ITEMS) - reales} averías; {sum(len(i['audio']) for i in ITEMS) / 16000 / 60:.1f} min de audio")

In [ ]:
# 4. Cada configuración las oye en su propio proceso. Si una falla (p. ej. sin memoria), se apunta y se sigue
RES = {}
for n, (nombre, modelo, marcas, lote) in enumerate(CONFIGURACIONES):
    salida = f"/content/whisper_{n}.json"
    print(f"→ {nombre}…", flush=True)
    r = subprocess.run([sys.executable, "/content/repo/fabrica/comparar_whisper.py", "oir", modelo, marcas,
                        "/content/items_whisper.pkl", salida, str(lote)], capture_output=True, text=True)
    lineas = [l for l in (r.stdout + r.stderr).splitlines() if l.strip() and "Warning" not in l]
    if r.returncode == 0 and Path(salida).exists():
        RES[nombre] = json.loads(Path(salida).read_text(encoding="utf-8"))
        print("  ", lineas[-1] if lineas else "hecho")
    else:
        print(f"   FALLÓ ({r.returncode}). Últimas líneas:")
        print("\n".join("     " + l for l in lineas[-8:]))

In [ ]:
# 5. Comparar y dejar el resultado en naval-podcast/pruebas/whisper_turbo.md
texto, VEREDICTOS = cw.informe(ITEMS, RES, CFG["verificacion"])
destino = RAIZ_EPISODIOS.parent / "pruebas" / "whisper_turbo.md"
destino.parent.mkdir(parents=True, exist_ok=True)
destino.write_text(texto, encoding="utf-8")
print(texto.split("\n## ")[0])
print("RESULTADO:", " · ".join(f"{k}: {'VALE' if v else 'no vale'}" for k, v in VEREDICTOS.items()), "→", destino)

In [ ]:
# 6. Subir a Drive y desconectar
comun.desconectar()